In [24]:
import random
import itertools
import math
import time
import os

In [25]:
def fill_rooms(size, n_rooms):
    grid = []
    for row in range(size):
        grid.append([])
        for col in range(size):
            grid[row].append(0)
    for room in range(1, n_rooms+1):
        cell = select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = True)
        grid[cell[0]][cell[1]] = room
        direction = random.randint(1,4)
        if direction == 1 and cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] == 0:
                grid[cell[0]-1][cell[1]] = room
            else: 
                direction = 2
        if direction == 2 and cell[0] < size - 1: 
            if grid[cell[0]+1][cell[1]] == 0:
                grid[cell[0]+1][cell[1]] = room
            else: 
                direction = 3
        if direction == 3 and cell[1] > 0: 
            if grid[cell[0]][cell[1]-1] == 0:
                grid[cell[0]][cell[1]-1] = room
            else: 
                direction = 4
        if direction == 4 and cell[1] < size - 1: 
            if grid[cell[0]][cell[1]+1] == 0:
                grid[cell[0]][cell[1]+1] = room
            else: 
                direction = 1
        if direction == 1 and cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] == 0:
                grid[cell[0]-1][cell[1]] = room

    while 0 in list(set(itertools.chain(*grid))):
        cell = select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = True)
        neighbor_rooms = []
        if cell[0] > 0: 
            if grid[cell[0]-1][cell[1]] != 0:
                neighbor_rooms.append(grid[cell[0]-1][cell[1]])
        if cell[0] < size - 1: 
            if grid[cell[0]+1][cell[1]] != 0:
                neighbor_rooms.append(grid[cell[0]+1][cell[1]])
        if cell[1] > 0: 
            if grid[cell[0]][cell[1]-1] != 0:
                neighbor_rooms.append(grid[cell[0]][cell[1]-1])
        if cell[1] < size - 1: 
            if grid[cell[0]][cell[1]+1] != 0:
                neighbor_rooms.append(grid[cell[0]][cell[1]+1])
        if len(neighbor_rooms) > 0:
            grid[cell[0]][cell[1]] = random.choice(neighbor_rooms)
    return grid

def select_random_cell_selecting_rooms(size, grid, cannot_have_room_selected = False):
    if not cannot_have_room_selected:
        return [random.randint(0,size-1), random.randint(0,size-1)]
    else:
        while 0 in list(set(itertools.chain(*grid))):
            cell = [random.randint(0,size-1), random.randint(0,size-1)]
            if grid[cell[0]][cell[1]] == 0:
                return cell
        return [-1, -1]

def find_room_locations(grid, room_num):
    room_locations = []
    for row in range(len(grid)):
        for col in range(len(grid[row])):
            if grid[row][col] == room_num:
                room_locations.append([row, col])
    return room_locations


def count_people_in_room(rooms, room_num, person_positions):
    n_people = 0
    #print(person_positions)
    for [i, j] in person_positions:
        if rooms[i][j] == room_num:
            n_people += 1
    return n_people


def place_persons(size, rooms):
    not_finished = True
    room_with_victim = 0
    room_nums = list(set(itertools.chain(*rooms)))
    num_tries = 0
    person_positions = []
    while not_finished and len(room_nums) > 1 and num_tries < 1000:
        person_positions = []
        for row in range(0, size):
            double_col = True
            while double_col:
                double_col = False
                col = random.randint(0, size-1)
                for [i, j] in person_positions:
                    if j == col: 
                        double_col = True
            person_positions.append([row, col])
        for room_num in room_nums:
            if count_people_in_room(rooms, room_num, person_positions) == 2:
                not_finished = False
                room_with_victim = room_num
        num_tries += 1
    if num_tries == 1000:
        return []
    not_found_victim = True
    random.shuffle(person_positions)
    for [i, j] in person_positions:
        if room_with_victim == rooms[i][j] and not_found_victim:
            not_found_victim = False
            person_positions.remove([i, j])
            person_positions.append([i, j])
    return person_positions


def place_objects(size, rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate):
    object_grid = []
    for row in range(size):
        object_grid.append([])
        for col in range(size):
            object_grid[row].append(0)

    for i in range(math.floor(object_rate * len(rooms) * len(rooms))):
        object_type = random.randint(-objects_cannot_be_occupied, objects_can_be_occupied-1)  #objects not occupied get negative index, objects that can be occupied positive
        can_be_occupied = False
        if object_type >= 0:
            can_be_occupied = True
            object_type += 1
        size_hor = 1 
        size_ver = 1
        if object_type == 1 or object_type == -1:
            if random.random() < 0.5:
                size_hor = 2
            else:
                size_ver = 2
            i += 1
        if object_type == 2:
            size_hor = random.randint(1,3)
            size_ver = random.randint(1,3)
            i += size_hor * size_ver - 1

        object_position = select_random_cell_for_object(len(rooms), rooms, person_positions, object_grid, can_be_occupied, size_hor, size_ver)
        if object_position != [-1, -1]:
            for i in range(size_hor):
                for j in range(size_ver):
                    object_grid[object_position[0]+i][object_position[1]+j] = object_type
        
    return object_grid


def select_random_cell_for_object(size, rooms, person_positions, object_grid, can_be_occupied, size_hor = 1, size_ver = 1):
    num_tries = 1
    if can_be_occupied:
        occupied_or_different_rooms = False
        cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
        room = rooms[cell[0]][cell[1]]
        for i in range(size_hor):
            for j in range(size_ver):
                occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0
        while occupied_or_different_rooms:
            occupied_or_different_rooms = False
            cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
            room = rooms[cell[0]][cell[1]]
            for i in range(size_hor):
                for j in range(size_ver):
                    occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0
            num_tries += 1
            if num_tries > 1000:
                return [-1, -1]
        return cell
    
    else:
        occupied_or_different_rooms = False
        cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
        room = rooms[cell[0]][cell[1]]
        for i in range(size_hor):
            for j in range(size_ver):
                occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0 or [cell[0]+i, cell[1]+j] in person_positions
        while occupied_or_different_rooms:
            occupied_or_different_rooms = False
            cell = [random.randint(0,size-size_hor), random.randint(0,size-size_ver)]
            room = rooms[cell[0]][cell[1]]
            for i in range(size_hor):
                for j in range(size_ver):
                    occupied_or_different_rooms = occupied_or_different_rooms or rooms[cell[0]+i][cell[1]+j] != room or object_grid[cell[0]+i][cell[1]+j] != 0 or [cell[0]+i, cell[1]+j] in person_positions
            num_tries += 1
            if num_tries > 1000:
                return [-1, -1]
        return cell

def gender_persons(size):
    genders = ["female", "male"]
    return random.choices(genders, k=size)


In [ ]:
def generate_clues(size, rooms, person_positions, object_positions, person_genders):
    clues = []
    for pers in range(size - 1):
        clue = generate_clue(size, rooms, person_positions, object_positions, person_genders, pers)
        clues.append(clue)
    number_solutions = check_number_solutions(size, rooms, object_positions, person_genders, clues)
    LIMIT = 3000
    sols = []
    n = check_number_solutions(size, rooms, object_positions, person_genders, clues, LIMIT, sols)
    while n >= LIMIT:                       # phase 1: space too big, add clues blindly
        for pers in range(size - 1):
            clues.append(generate_clue(size, rooms, person_positions, object_positions, person_genders, pers))
        sols = []
        n = check_number_solutions(size, rooms, object_positions, person_genders, clues, LIMIT, sols)

    while len(sols) > 1:                    # phase 2: pure filtering
        pers = random.randint(0, size - 2)
        clue = generate_clue(size, rooms, person_positions, object_positions, person_genders, pers)
        new_sols = [p for p in sols if check_clue(size, rooms, p, object_positions, person_genders, clue)]
        if len(new_sols) < len(sols):
            clues.append(clue)
            sols = new_sols
    clues_filtered = []
    for clue in clues:
        if not 'pers' in clue:
            clues_filtered.append(clue)
    for pers in range(size - 1):
        clues_filtered.extend(filter_clues(size, rooms, object_positions, person_genders, clues, pers))
 #       print(clues_filtered)
    return clues_filtered

def filter_clues(size, rooms, object_positions, person_genders, clues, pers):
    clues_pers = get_clues_pers(clues, pers)
    new_clues_pers = clues_pers.copy()
    
    for clue in reversed(clues_pers):
        if not "pers2" in clue and clue["type"] in ['next_to', 'not_next_to', 'on', 'in', 'in_top_row', 'in_bottom_row', 'in_left_column', 'in_right_column', 'not_in', 'in_corner', 'not_in_corner', 'not_on', 'in_column', 'in_row']:
            new_clues_pers2 = new_clues_pers.copy()
            new_clues_pers2.remove(clue)
            possible_cells = get_possible_cells_pers(size, rooms, object_positions, person_genders, new_clues_pers, pers)
            possible_cells_ex_clue = get_possible_cells_pers(size, rooms, object_positions, person_genders, new_clues_pers2, pers)
            if len(new_clues_pers2) > 0 and len(possible_cells) == len(possible_cells_ex_clue):
                new_clues_pers.remove(clue)
    return new_clues_pers

def generate_clue(size, rooms, person_positions, object_positions, person_genders, pers):
    not_found = True
    cell = person_positions[pers]
    room = rooms[cell[0]][cell[1]]

    while not_found:
        types =   8 * ['in_same_room', 'not_in_same_room', 'next_to', 'not_next_to', 'on', 'in'] \
                + 4 * ['in_top_row', 'in_bottom_row', 'in_left_column', 'in_right_column', 'alone', 'not_alone', 'not_in',  'in_corner', 'not_in_corner', 'no_empty_room'] \
                + 2 * ['in_room_with_female', 'not_in_room_with_female', 'in_room_with_male', 'not_in_room_with_male', 'not_on', 'only_one_on',  'n_persons_on', 'in_room_with_multiple_females', 'in_room_with_multiple_males'] \
                + 1 * ['in_column', 'in_row', 'east_of', 'west_of', 'north_of', 'south_of', 'next_to_with_someone_else', 'object_in_col', 'no_object_in_col', 'object_in_row', 'no_object_in_row', 'someone_else_in_room_on']
        type = random.choice(types)
        clue = {'type': type, 'pers': pers}

        if type == 'in_same_room':
            pers2list = []
            for pers2 in range(size - 1):
                if pers2 != pers:
                    cell2 = person_positions[pers2]
                    room2 = rooms[cell2[0]][cell2[1]]
                    if room == room2:
                        pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'not_in_same_room':
            pers2list = []
            for pers2 in range(size - 1):
                if pers2 != pers:
                    cell2 = person_positions[pers2]
                    room2 = rooms[cell2[0]][cell2[1]]
                    if room != room2:
                        pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'east_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[1] > cell2[1]:
                    pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'west_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[1] < cell2[1]:
                    pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'north_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[0] < cell2[0]:
                    pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'south_of':
            pers2list = []
            for pers2 in range(size - 1):
                cell2 = person_positions[pers2]
                if pers2 != pers and cell[0] > cell2[0]:
                    pers2list.append(pers2)
            if len(pers2list) > 0:
                clue["pers2"] = random.choice(pers2list)
                return clue
        if type == 'next_to':
            objectlist = []
            if cell[0] < size - 1 and object_positions[cell[0]+1][cell[1]] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]+1][cell[1]]:
                objectlist.append(object_positions[cell[0]+1][cell[1]])
            if cell[0] > 0 and object_positions[cell[0]-1][cell[1]] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]-1][cell[1]]:
                objectlist.append(object_positions[cell[0]-1][cell[1]])
            if cell[1] < size - 1 and object_positions[cell[0]][cell[1]+1] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]+1]:
                objectlist.append(object_positions[cell[0]][cell[1]+1])
            if cell[1] > 0 and object_positions[cell[0]][cell[1]-1] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]-1]:
                objectlist.append(object_positions[cell[0]][cell[1]-1])
            if len(objectlist) > 0:
                clue["obj"] = random.choice(objectlist)
                return clue
        if type == 'not_next_to':            
            objectlist = [0]
            if cell[0] < size - 1 and object_positions[cell[0]+1][cell[1]] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]+1][cell[1]]:
                objectlist.append(object_positions[cell[0]+1][cell[1]])
            if cell[0] > 0 and object_positions[cell[0]-1][cell[1]] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]-1][cell[1]]:
                objectlist.append(object_positions[cell[0]-1][cell[1]])
            if cell[1] < size - 1 and object_positions[cell[0]][cell[1]+1] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]+1]:
                objectlist.append(object_positions[cell[0]][cell[1]+1])
            if cell[1] > 0 and object_positions[cell[0]][cell[1]-1] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]-1]:
                objectlist.append(object_positions[cell[0]][cell[1]-1])
            notobjectlist = [x for x in list(set(itertools.chain(*object_positions))) if x not in objectlist]
            if len(notobjectlist) > 0:
                clue["obj"] = random.choice(notobjectlist)
                return clue
        if type == 'on':
            if object_positions[cell[0]][cell[1]] != 0:
                clue['obj'] = object_positions[cell[0]][cell[1]]
                return clue
        if type == 'not_on':
            if object_positions[cell[0]][cell[1]] != 0:
                objects = [x for x in list(set(itertools.chain(*object_positions))) if x > 0 and x != object_positions[cell[0]][cell[1]]]
                if len(objects) > 0:
                    clue["obj"] = random.choice(objects)
                    return clue
        if type == 'in':
            clue['room'] = rooms[cell[0]][cell[1]]
            return clue
        if type == 'not_in':
            room_nums = list(set(itertools.chain(*rooms)))
            room_nums.remove(rooms[cell[0]][cell[1]])
            clue['room'] = random.choice(room_nums) 
            return clue
        if type == 'in_corner':
            if in_corner(size, rooms, person_positions, pers):
                return clue
        if type == 'not_in_corner':
            if not_in_corner(size, rooms, person_positions, pers):
                return clue
        if type == 'in_room_with_female':
            if in_room_with_female(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'not_in_room_with_female':
            if not in_room_with_female(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'in_room_with_male':
            if in_room_with_male(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'not_in_room_with_male':
            if not in_room_with_male(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'alone':
            if not in_room_with_female(size, rooms, person_positions, person_genders, pers) and not in_room_with_male(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'not_alone':
            if in_room_with_female(size, rooms, person_positions, person_genders, pers) or in_room_with_male(size, rooms, person_positions, person_genders, pers):
                return clue
        if type == 'in_left_column':
            if cell[1] == 0:
                return clue
        if type == 'in_right_column':
            if cell[1] == size - 1:
                return clue
        if type == 'in_top_row':
            if cell[0] == 0:
                return clue
        if type == 'in_bottom_row':
            if cell[0] == size - 1:
                return clue
        if type == 'in_column':
            clue["n"] = cell[1]
            return clue
        if type == 'in_row':
            clue["n"] = cell[0]
            return clue
        if type == 'only_one_on':
            if object_positions[cell[0]][cell[1]] != 0:
                obj = object_positions[cell[0]][cell[1]]
                if only_one_on(person_positions, object_positions, pers, obj):
                    clue['obj'] = obj
                    return clue
        if type == 'in_room_with_multiple_females':
            if in_room_with_multiple_females(size, rooms, person_positions, person_genders, pers, 2):
                clue["n"] = 2
                return clue
        if type == 'in_room_with_multiple_males':
            if in_room_with_multiple_males(size, rooms, person_positions, person_genders, pers, 2):
                clue["n"] = 2
                return clue
        if type == 'next_to_with_someone_else':
            objectlist = []
            if cell[0] < size - 1 and object_positions[cell[0]+1][cell[1]] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]+1][cell[1]]:
                if next_to_with_someone_else(size, rooms, person_positions, object_positions, pers, object_positions[cell[0]+1][cell[1]]):
                    objectlist.append(object_positions[cell[0]+1][cell[1]])
            if cell[0] > 0 and object_positions[cell[0]-1][cell[1]] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]-1][cell[1]]:
                if next_to_with_someone_else(size, rooms, person_positions, object_positions, pers, object_positions[cell[0]-1][cell[1]]):
                    objectlist.append(object_positions[cell[0]-1][cell[1]])
            if cell[1] < size - 1 and object_positions[cell[0]][cell[1]+1] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]+1]:
                if next_to_with_someone_else(size, rooms, person_positions, object_positions, pers, object_positions[cell[0]][cell[1]+1]):
                    objectlist.append(object_positions[cell[0]][cell[1]+1])
            if cell[1] > 0 and object_positions[cell[0]][cell[1]-1] != 0 and rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]-1]:
                if next_to_with_someone_else(size, rooms, person_positions, object_positions, pers, object_positions[cell[0]][cell[1]-1]):
                    objectlist.append(object_positions[cell[0]][cell[1]-1])
            if len(objectlist) > 0:
                clue["obj"] = random.choice(objectlist)
                return clue
        if type == 'n_persons_on':
            objectlist = [x for x in list(set(itertools.chain(*object_positions))) if x > 0]
            if len(objectlist) > 0:
                obj = random.choice(objectlist)
                for n in range(len(person_positions)):
                    if n_persons_on(person_positions, object_positions, obj, n):
                        clue["obj"] = obj
                        clue["n"] = n
                        clue.pop("pers")
                        return clue
        if type == 'no_empty_room':
            if no_empty_room(rooms, person_positions):
                clue.pop("pers")
                return clue
        if type == 'object_in_col':
            objectlist = []
            col = cell[1]
            for i in range(size):
                if object_positions[i][col] < 0 and object_positions[i][col] not in objectlist:
                    objectlist.append(object_positions[i][col])
            if len(objectlist) > 0:
                clue["obj"] = random.choice(objectlist)
                return clue
        if type == 'no_object_in_col':
            objectlist = []
            not_objectlist = []
            col = cell[1]
            for i in range(size):
                if object_positions[i][col] < 0 and object_positions[i][col] not in objectlist:
                    objectlist.append(object_positions[i][col])
            not_objectlist = [x for x in list(set(itertools.chain(*object_positions))) if x < 0 and x not in objectlist]
            if len(not_objectlist) > 0:
                clue["obj"] = random.choice(not_objectlist)
                return clue
        if type == 'object_in_row':
            objectlist = []
            row = cell[0]
            for i in range(size):
                if object_positions[row][i] < 0 and object_positions[row][i] not in objectlist:
                    objectlist.append(object_positions[row][i])
            if len(objectlist) > 0:
                clue["obj"] = random.choice(objectlist)
                return clue
        if type == 'no_object_in_row':
            objectlist = []
            not_objectlist = []
            row = cell[0]
            for i in range(size):
                if object_positions[row][i] < 0 and object_positions[row][i] not in objectlist:
                    objectlist.append(object_positions[row][i])
            not_objectlist = [x for x in list(set(itertools.chain(*object_positions))) if x < 0 and x not in objectlist]
            if len(not_objectlist) > 0:
                clue["obj"] = random.choice(not_objectlist)
                return clue
        if type == 'someone_else_in_room_on':
            objectlist = []
            for pers2 in range(size):
                if pers2 != pers and in_same_room(rooms, person_positions, pers, pers2):
                    cell2 = person_positions[pers2]
                    if object_positions[cell2[0]][cell2[1]] != 0:
                        objectlist.append(object_positions[cell2[0]][cell2[1]])
            if len(objectlist) > 0:
                clue["obj"] = random.choice(objectlist)
                return clue


def get_clues_pers(clues, pers):
    clues_pers = []
    for clue in clues:
        if 'pers' in clue and clue['pers'] == pers:
            clues_pers.append(clue)
    return clues_pers

def get_possible_cells_pers(size, rooms, object_positions, person_genders, clues, pers):
    possible_cells = []
    clues_pers = get_clues_pers(clues, pers)
    for i in range(size):
        for j in range(size):
            person_positions_single_pers = size * [[0, 0]]
            person_positions_single_pers[pers] = [i, j]
            add_to_possibilities = True
            for clue in clues_pers:
                if "pers2" not in clue and clue['type'] in ['next_to', 'not_next_to', 'on', 'in', 'in_top_row', 'in_bottom_row', 'in_left_column', 'in_right_column', 'not_in', 'in_corner', 'not_in_corner', 'not_on', 'in_column', 'in_row']:
                    add_to_possibilities = add_to_possibilities and (object_positions[i][j] >= 0) and check_clue(size, rooms, person_positions_single_pers, object_positions, person_genders, clue)
            if add_to_possibilities:
                possible_cells.append([i, j])
    return possible_cells


class _Cap(Exception): pass

def check_number_solutions(size, rooms, object_positions, person_genders, clues, limit=None, collect=None):
    n_solutions = 0
    valid_cells = []
    
    for pers in range(size):
        valid_cells.append((pers, get_possible_cells_pers(size, rooms, object_positions, person_genders, clues, pers)))

    valid_cells.sort(key=lambda x: len(x[1]))

    placement_order = {}
    for index, (pers, _) in enumerate(valid_cells):
        placement_order[pers] = index

    clues_by_index = {i: [] for i in range(size + 1)}
    
    # These clues require iterating over all persons on the board, 
    # so they must wait until index == size (the leaf nodes)
    global_clue_types = [
        'no_empty_room', 'only_one_on', 'n_persons_on', 
        'in_room_with_multiple_females', 'in_room_with_multiple_males', 
        'next_to_with_someone_else', 'alone', 'not_alone',
        'in_room_with_female', 'not_in_room_with_female', 'in_room_with_male', 'not_in_room_with_male',
        'someone_else_in_room_on'
    ]
                         
    for clue in clues:
        if clue['type'] in global_clue_types:
            clues_by_index[size].append(clue) 
        elif 'pers2' in clue:
            max_index = max(placement_order[clue['pers']], placement_order[clue['pers2']])
            clues_by_index[max_index].append(clue)
        else:
            clues_by_index[placement_order[clue['pers']]].append(clue)

    used_rows = [False] * size
    used_cols = [False] * size

    # Maintain a running position list instead of generating test configurations at the end
    # Initialize with [-1, -1] so partial checks don't hit IndexErrors
    running_positions = [[-1, -1] for _ in range(size)]
    def backtrack(index):
        nonlocal n_solutions
        
        if index == size:
            # Check victim logic (victim is always person index size-1)
            pos_victim = running_positions[size-1]
            room_victim = rooms[pos_victim[0]][pos_victim[1]]
            if count_people_in_room(rooms, room_victim, running_positions) != 2:
                return

            # Evaluate remaining global clues
            for clue in clues_by_index[size]:
                if not check_clue(size, rooms, running_positions, object_positions, person_genders, clue):
                    return
            
            n_solutions += 1
            if collect is not None:
                collect.append([p[:] for p in running_positions])
            if limit is not None and n_solutions >= limit:
                raise _Cap()
            return

        pers, possible_cells = valid_cells[index]

        for r, c in possible_cells:
            if not used_rows[r] and not used_cols[c]:
                # 1. Place person
                used_rows[r] = True
                used_cols[c] = True
                running_positions[pers] = [r, c]

                # 2. FORWARD PRUNING: Do all future persons still have at least one free cell?
                future_is_viable = True
                for next_index in range(index + 1, size):
                    next_pers, next_possible_cells = valid_cells[next_index]
                    # Check if next_pers has ANY cell where row and col are both unused
                    if not any(not used_rows[nr] and not used_cols[nc] for nr, nc in next_possible_cells):
                        future_is_viable = False
                        break
                
                # 3. INCREMENTAL CLUE CHECK: Only check clues if the grid is still viable
                if future_is_viable:
                    valid_so_far = True
                    for clue in clues_by_index[index]:
                        if not check_clue(size, rooms, running_positions, object_positions, person_genders, clue):
                            valid_so_far = False
                            break
                    
                    # 4. Branch further
                    if valid_so_far:
                        backtrack(index + 1)

                # 5. Remove person (backtrack step)
                used_rows[r] = False
                used_cols[c] = False
                running_positions[pers] = [-1, -1]

    try:
        backtrack(0)
    except _Cap:
        pass
    return n_solutions

def check_solution(size, rooms, object_positions, person_genders, clues, person_positions_test, printstats = False):
    is_solution = True
    n_drops = len(clues) * [0]
    for i in range(size):
        row = person_positions_test[i][0]
        col = person_positions_test[i][1]
        is_solution = is_solution and (object_positions[row][col] >= 0)
    if printstats:
        print("person ", i, " on object:", is_solution)
    pos_victim = person_positions_test[size-1]
    room_victim = rooms[pos_victim[0]][pos_victim[1]]
    people_in_room_victim = count_people_in_room(rooms, room_victim, person_positions_test)
    is_solution = is_solution and (people_in_room_victim == 2)
    if printstats:
        print("victim in room with 1 other:", is_solution, ", because ", people_in_room_victim, " in room.")

    if len(clues) > 0:
        for i, clue in enumerate(clues):
            is_solution = is_solution and check_clue(size, rooms, person_positions_test, object_positions, person_genders, clue)
            if is_solution: 
                n_drops[i] += 1
            if printstats:
                print("clue ", i, clue, ": ", is_solution)
    return is_solution

In [27]:
def check_clue(size, rooms, person_positions, object_positions, person_genders, clue):
    type = clue["type"]
    if type == 'in_same_room':
        return in_same_room(rooms, person_positions, clue["pers"], clue["pers2"])
    if type == 'not_in_same_room':
        return not_in_same_room(rooms, person_positions, clue["pers"], clue["pers2"])
    if type == 'next_to':
        return next_to(size, rooms, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'not_next_to':
        return not_next_to(size, rooms, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'on':
        return on(person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'not_on':
        return not_on(person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'in':
        return in_room(rooms, person_positions, clue["pers"], clue["room"])
    if type == 'not_in':
        return not_in_room(rooms, person_positions, clue["pers"], clue["room"])
    if type == 'in_room_with_female':
        return in_room_with_female(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'not_in_room_with_female':
        return not_in_room_with_female(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'in_room_with_male':
        return in_room_with_male(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'not_in_room_with_male':
        return not_in_room_with_male(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'alone':
        return alone(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'not_alone':
        return not_alone(size, rooms, person_positions, person_genders, clue["pers"])
    if type == 'in_corner':
        return in_corner(size, rooms, person_positions, clue["pers"])
    if type == 'not_in_corner':
        return not_in_corner(size, rooms, person_positions, clue["pers"])
    if type == 'in_left_column':
        return in_left_column(size, person_positions, clue["pers"])
    if type == 'in_right_column':
        return in_right_column(size, person_positions, clue["pers"])
    if type == 'in_top_row':
        return in_top_row(size, person_positions, clue["pers"])
    if type == 'in_bottom_row':
        return in_bottom_row(size, person_positions, clue["pers"])
    if type == 'in_column':
        return in_column(person_positions, clue["pers"], clue["n"])
    if type == 'in_row':
        return in_row(person_positions, clue["pers"], clue["n"])
    if type == 'east_of':
        return east_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'west_of':
        return west_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'north_of':
        return north_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'south_of':
        return south_of(person_positions, clue["pers"], clue["pers2"])
    if type == 'only_one_on':
        return only_one_on(person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'n_persons_on':
        return n_persons_on(person_positions, object_positions, clue["obj"], clue["n"])
    if type == 'no_empty_room':
        return no_empty_room(rooms, person_positions)
    if type == 'in_room_with_multiple_females':
        return in_room_with_multiple_females(size, rooms, person_positions, person_genders, clue["pers"], clue["n"])
    if type == 'in_room_with_multiple_males':
        return in_room_with_multiple_males(size, rooms, person_positions, person_genders, clue["pers"], clue["n"])
    if type == 'next_to_with_someone_else':
        return next_to_with_someone_else(size, rooms, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'object_in_col':
        return object_in_col(size, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'no_object_in_col':
        return no_object_in_col(size, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'object_in_row':
        return object_in_row(size, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'no_object_in_row':
        return no_object_in_row(size, person_positions, object_positions, clue["pers"], clue["obj"])
    if type == 'someone_else_in_room_on':
        return someone_else_in_room_on(size, rooms, person_positions, object_positions, clue["pers"], clue["obj"])
    return False


def in_same_room(rooms, person_positions, pers, pers2):
    cellA = person_positions[pers]
    cellB = person_positions[pers2]
#    print(cellA, cellB, rooms[cellA[0]][cellA[1]], rooms[cellB[0]][cellB[1]])
    return rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]

def not_in_same_room(rooms, person_positions, pers, pers2):
    return not in_same_room(rooms, person_positions, pers, pers2)

def next_to(size, rooms, person_positions, object_positions, pers, obj):
    cell = person_positions[pers]
    next_to = False
    if cell[0] < size - 1:
        next_to = next_to or rooms[cell[0]][cell[1]] == rooms[cell[0]+1][cell[1]] and object_positions[cell[0]+1][cell[1]] == obj 
    if cell[0] > 0:
        next_to = next_to or rooms[cell[0]][cell[1]] == rooms[cell[0]-1][cell[1]] and object_positions[cell[0]-1][cell[1]] == obj
    if cell[1] < size - 1:
        next_to = next_to or rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]+1] and object_positions[cell[0]][cell[1]+1] == obj
    if cell[1] > 0:
        next_to = next_to or rooms[cell[0]][cell[1]] == rooms[cell[0]][cell[1]-1] and object_positions[cell[0]][cell[1]-1] == obj
    return next_to

def not_next_to(size, rooms, person_positions, object_positions, pers, obj):
    return not next_to(size, rooms, person_positions, object_positions, pers, obj)

def on(person_positions, object_positions, pers, obj):
    cell = person_positions[pers]
    return object_positions[cell[0]][cell[1]] == obj

def not_on(person_positions, object_positions, pers, obj):
    return not on(person_positions, object_positions, pers, obj)

def in_corner(size, rooms, person_positions, pers):
    cell = person_positions[pers]
    against_vertical_wall = False
    if cell[1] == 0 or cell[1] == size - 1:
        against_vertical_wall = True
    else:
        if rooms[cell[0]][cell[1]] != rooms[cell[0]][cell[1]-1] or rooms[cell[0]][cell[1]] != rooms[cell[0]][cell[1]+1]:
            against_vertical_wall = True
    against_horizontal_wall = False
    if cell[0] == 0 or cell[0] == size - 1:
        against_horizontal_wall = True
    else:
        if rooms[cell[0]][cell[1]] != rooms[cell[0]-1][cell[1]] or rooms[cell[0]][cell[1]] != rooms[cell[0]+1][cell[1]]:
            against_horizontal_wall = True
    return against_horizontal_wall and against_vertical_wall

def not_in_corner(size, rooms, person_positions, pers):
    return not in_corner(size, rooms, person_positions, pers)

def in_room(rooms, person_positions, pers, room):
    cell = person_positions[pers]
    return rooms[cell[0]][cell[1]] == room

def not_in_room(rooms, person_positions, pers, room):
    return not in_room(rooms, person_positions, pers, room)

def in_room_with_female(size, rooms, person_positions, person_genders, pers):
    cellA = person_positions[pers]
    for i in range(size):
        cellB = person_positions[i]
        if i != pers and person_genders[i] == 'female' and rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]:
            return True
    return False

def not_in_room_with_female(size, rooms, person_positions, person_genders, pers):
    return not in_room_with_female(size, rooms, person_positions, person_genders, pers)

def in_room_with_male(size, rooms, person_positions, person_genders, pers):
    cellA = person_positions[pers]
    for i in range(size):
        cellB = person_positions[i]
        if i != pers and person_genders[i] == 'male' and rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]:
            return True
    return False

def not_in_room_with_male(size, rooms, person_positions, person_genders, pers):
    return not in_room_with_male(size, rooms, person_positions, person_genders, pers)

def alone(size, rooms, person_positions, person_genders, pers):
    return not_in_room_with_female(size, rooms, person_positions, person_genders, pers) and not_in_room_with_male(size, rooms, person_positions, person_genders, pers)

def not_alone(size, rooms, person_positions, person_genders, pers):
    return in_room_with_female(size, rooms, person_positions, person_genders, pers) or in_room_with_male(size, rooms, person_positions, person_genders, pers)

def in_left_column(size, person_positions, pers):
    return in_column(person_positions, pers, 0)

def in_right_column(size, person_positions, pers):
    return in_column(person_positions, pers, size - 1)

def in_top_row(size, person_positions, pers):
    return in_row(person_positions, pers, 0)

def in_bottom_row(size, person_positions, pers):
    return in_row(person_positions, pers, size - 1)

def in_column(person_positions, pers, n):
    cell = person_positions[pers]
    return cell[1] == n

def in_row(person_positions, pers, n):
    cell = person_positions[pers]
    return cell[0] == n

def east_of(person_positions, pers, pers2):
    cellA = person_positions[pers]
    cellB = person_positions[pers2]
    return cellA[1] > cellB[1] 

def west_of(person_positions, pers, pers2):
    return not east_of(person_positions, pers, pers2)

def north_of(person_positions, pers, pers2):
    cellA = person_positions[pers]
    cellB = person_positions[pers2]
    return cellA[0] < cellB[0] 

def south_of(person_positions, pers, pers2):
    return not north_of(person_positions, pers, pers2)

def only_one_on(person_positions, object_positions, pers, obj):
    return on(person_positions, object_positions, pers, obj) and n_persons_on(person_positions, object_positions, obj, 1)

def n_persons_on(person_positions, object_positions, obj, n):
    n_on = 0
    for pers in range(len(person_positions)):
        if on(person_positions, object_positions, pers, obj):
            n_on += 1
    return n_on == n

def no_empty_room(rooms, person_positions):
    room_nums = list(set(itertools.chain(*rooms)))
    has_empty_room = False
    for room_num in room_nums:
        if count_people_in_room(rooms, room_num, person_positions) == 0:
            has_empty_room = True
    return not has_empty_room

def in_room_with_multiple_females(size, rooms, person_positions, person_genders, pers, n):
    cellA = person_positions[pers]
    n_females = 0
    for pers2 in range(size):
        cellB = person_positions[pers2]
        if pers2 != pers and person_genders[pers2] == 'female' and rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]:
            n_females += 1
    return n_females == n

def in_room_with_multiple_males(size, rooms, person_positions, person_genders, pers, n):
    cellA = person_positions[pers]
    n_males = 0
    for pers2 in range(size):
        cellB = person_positions[pers2]
        if pers2 != pers and person_genders[pers2] == 'male' and rooms[cellA[0]][cellA[1]] == rooms[cellB[0]][cellB[1]]:
            n_males += 1
    return n_males == n

def next_to_with_someone_else(size, rooms, person_positions, object_positions, pers, obj):
    if next_to(size, rooms, person_positions, object_positions, pers, obj):
        objlist = []
        for i in range(len(object_positions)):
            for j in range(len(object_positions[i])):
                if object_positions[i][j] == obj:
                    objlist.append([i, j])
        if len(objlist) > 0:
            for object in objlist:
                new_object_positions = []
                for i in range(size):
                    new_object_positions.append([])
                    for j in range(size):
                        new_object_positions[i].append(0)
                new_object_positions[object[0]][object[1]] = obj
                for pers2 in range(size):
                    if pers != pers2 and next_to(size, rooms, person_positions, new_object_positions, pers, obj) and next_to(size, rooms, person_positions, new_object_positions, pers2, obj):
                        return True
        return False
    else:
        return False

def object_in_col(size, person_positions, object_positions, pers, obj):
    col = person_positions[pers][1]
    for i in range(size):
        if object_positions[i][col] == obj:
            return True
    return False

def no_object_in_col(size, person_positions, object_positions, pers, obj):
    return not object_in_col(size, person_positions, object_positions, pers, obj)

def object_in_row(size, person_positions, object_positions, pers, obj):
    row = person_positions[pers][0]
    for i in range(size):
        if object_positions[row][i] == obj:
            return True
    return False

def no_object_in_row(size, person_positions, object_positions, pers, obj):
    return not object_in_row(size, person_positions, object_positions, pers, obj)

def someone_else_in_room_on(size, rooms, person_positions, object_positions, pers, obj):
    for pers2 in range(size):
        if pers2 != pers and in_same_room(rooms, person_positions, pers, pers2) and on(person_positions, object_positions, pers2, obj):
            return True
    return False

In [28]:
def convert_to_clingo(size, rooms, person_genders, object_positions, clues):

    puzzle_text = "#const n = "+ str(size-1) + "."
    puzzle_text += "\ncell(R,C) :- R = 0..n, C = 0..n."

    room_names = ['bedroom', 'livingroom', 'kitchen', 'bathroom', 'hallway', 'garage', 'diningroom', 'office', 'outside', 'garderobe', 'guestroom', 'attic', 'gameroom', 'panicroom']

    objects_cannot_be_occupied = ['table', 'plant', 'closet', 'box', 'statue']
    objects_can_be_occupied = ['sofa', 'carpet', 'chair', 'bed', 'deskchair']

    names_male = ['alphonse', 'bernard', 'charlie', 'daniel', 'edward', 'frank', 'george', 'henry', 'ivan', 'jesse', 'keith', 'louis', 'mike', 'victor']
    names_female = ['alice', 'bonnie', 'carol', 'daisy', 'emily', 'florence', 'grace', 'hannah', 'isla', 'jane', 'kate', 'lily', 'mary', 'victoria']


    puzzle_text += "\n\n%%%% Rooms"
    for room in list(set(itertools.chain(*rooms))):
        puzzle_text += "\nroom("+room_names[room-1]+")."

    for i, row in enumerate(rooms):
        for j, val in enumerate(row):
            puzzle_text += "\nin_room("+str(i)+","+str(j)+","+room_names[val-1]+")."


    puzzle_text += "\n\n%%%% Objects"
    for object in list(set(itertools.chain(*object_positions))):
        if(object < 0):
            puzzle_text += "\nobject("+objects_cannot_be_occupied[-object+1]+")."
            puzzle_text += "\ncan_not_be_occupied("+objects_cannot_be_occupied[-object+1]+")."
        if(object > 0):
            puzzle_text += "\nobject("+objects_can_be_occupied[object-1]+")."


    for i, row in enumerate(object_positions):
        for j, val in enumerate(row):
            if object_positions[i][j] < 0:
                puzzle_text += "\ncontains("+str(i)+","+str(j)+","+objects_cannot_be_occupied[-val+1]+")."
            if object_positions[i][j] > 0:
                puzzle_text += "\ncontains("+str(i)+","+str(j)+","+objects_can_be_occupied[val-1]+")."


    puzzle_text += "\n\n%%%% Suspects and victims"
    for i in range(size - 1):
        if person_genders[i] == 'female':
            puzzle_text += "\nsuspect("+names_female[i]+")."
            puzzle_text += "\nfemale("+names_female[i]+")."
        else:
            puzzle_text += "\nsuspect("+names_male[i]+")."
            puzzle_text += "\nmale("+names_male[i]+")."
    if person_genders[size - 1] == 'female':
        puzzle_text += "\nvictim("+names_female[-1]+")."
        puzzle_text += "\nfemale("+names_female[-1]+")."
    else:
        puzzle_text += "\nvictim("+names_male[-1]+")."
        puzzle_text += "\nmale("+names_male[-1]+")."
    
    clue_text = "%%%% Clues"
    for clue in clues:
        clue_string = ""
        for x, y in clue.items():
            if x != 'pers' and x != 'type':
                ytext = str(y)
                if x == 'obj':
                    if y < 0:
                        ytext = objects_cannot_be_occupied[-y+1]
                    elif y > 0:
                        ytext = objects_can_be_occupied[y-1]
                elif x == 'pers2':
                    if person_genders[y] == 'female':
                        ytext = names_female[y]
                    else:
                        ytext = names_male[y]
                elif x == 'room':
                    ytext = room_names[y-1]
                clue_string = clue_string + "," + ytext
        if 'pers' in clue:
            if person_genders[clue['pers']] == 'female':
                person_name = names_female[clue['pers']]
            else:
                person_name = names_male[clue['pers']]
            clue_text += "\nclue("+person_name+","+clue['type']+clue_string+")."
        else:
            clue_text += "\nclue("+clue['type']+clue_string+")."

    return puzzle_text, clue_text



In [29]:
start = time.time()
random.seed(11)

size = 4

n_rooms = random.randint(max(2, size - 3), size)
objects_can_be_occupied = 4    #The first is always size 2x1, the second will be randint(1,3) x randint(1,3)
objects_cannot_be_occupied = 3 #The first is always size 2x1
object_rate = 0.25

rooms = fill_rooms(size, n_rooms) 
person_positions = place_persons(size, rooms)



while len(person_positions) == 0:
    n_rooms = random.randint(size - 3, size)
    rooms = fill_rooms(size, n_rooms) 
    person_positions = place_persons(size, rooms)


start = time.time()

random.seed(11)
objects_can_be_occupied = 4    #The first is always size 2x1, the second will be randint(1,3) x randint(1,3)
objects_cannot_be_occupied = 3 #The first is always size 2x1
object_rate = 0.25

n_rooms = random.randint(size - 3, size)
rooms = fill_rooms(size, n_rooms) 
person_positions = place_persons(size, rooms)

while len(person_positions) == 0:
    n_rooms = random.randint(size - 3, size)
    rooms = fill_rooms(size, n_rooms) 
    person_positions = place_persons(size, rooms)



print(f"Number of rooms: {n_rooms}")
for row in range(size):
    print(rooms[row])

print('\nLocation persons:')
print(person_positions)
for i in range(size):
    line = '['
    for j in range(size):
        cell = '0'
        for k in range(size):
            if person_positions[k][0] == i and person_positions[k][1] == j:
                cell = str(k)
        if j == size - 1:
            cell = cell + ']'
        else: 
            cell = cell + ', '
        line = line + cell
    print(line)

print('\nGenders:')
person_genders = gender_persons(size)
print(person_genders)

print('\nMap of objects:')
object_positions = place_objects(size, rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate)
for row in range(size):
    print(object_positions[row])

if True:
    print('\nClues:')
    clues = generate_clues(size, rooms, person_positions, object_positions, person_genders)
    for clue in clues:
        print(clue)

    nsol = check_number_solutions(size, rooms, object_positions, person_genders, clues)
    print(f'\nSolutions: {nsol}')
else:
    clues = [{'type': 'not_on', 'pers': 0, 'obj': 2},\
             {'type': 'north_of', 'pers': 1, 'pers2': 0},\
                {'type': 'next_to', 'pers': 2, 'obj': -2},\
                    {'type': 'not_in_corner', 'pers': 3},\
                        {'type': 'in_corner', 'pers': 4},\
                            {'type': 'not_in_room', 'pers': 5, 'room': 2},\
                                {'type': 'next_to', 'pers': 4, 'obj': -1},\
                                    {'type': 'in_room', 'pers': 0, 'room': 5},\
                                        {'type': 'in_room', 'pers': 4, 'room': 3}]
    print(check_solution(size, rooms, object_positions, person_genders, clues, person_positions, printstats = True))


end = time.time()
print(end - start, "seconds")


puzzle_text, clues_text = convert_to_clingo(size, rooms, person_genders, object_positions, clues)

print(puzzle_text)
print(clues_text)
    

Number of rooms: 4
[4, 4, 3, 3]
[4, 4, 4, 2]
[4, 4, 2, 2]
[4, 4, 4, 1]

Location persons:
[[3, 3], [2, 2], [1, 1], [0, 0]]
[3, 0, 0, 0]
[0, 2, 0, 0]
[0, 0, 1, 0]
[0, 0, 0, 0]

Genders:
['male', 'female', 'female', 'male']

Map of objects:
[2, 2, -3, 0]
[0, 0, 0, 0]
[0, -2, 0, 0]
[0, -1, -1, 0]

Clues:
{'type': 'in', 'pers': 0, 'room': 1}
{'type': 'west_of', 'pers': 1, 'pers2': 0}
{'type': 'in', 'pers': 1, 'room': 2}
{'type': 'next_to', 'pers': 2, 'obj': -2}

Solutions: 1
0.008374691009521484 seconds
#const n = 3.
cell(R,C) :- R = 0..n, C = 0..n.

%%%% Rooms
room(bedroom).
room(livingroom).
room(kitchen).
room(bathroom).
in_room(0,0,bathroom).
in_room(0,1,bathroom).
in_room(0,2,kitchen).
in_room(0,3,kitchen).
in_room(1,0,bathroom).
in_room(1,1,bathroom).
in_room(1,2,bathroom).
in_room(1,3,livingroom).
in_room(2,0,bathroom).
in_room(2,1,bathroom).
in_room(2,2,livingroom).
in_room(2,3,livingroom).
in_room(3,0,bathroom).
in_room(3,1,bathroom).
in_room(3,2,bathroom).
in_room(3,3,bedroom).



In [ ]:
output_dir = 'extraction/autogenerated_puzzles'

for size in range(8,11):
    for seed in range(100):

        start = time.time()

        random.seed(seed)
        objects_can_be_occupied = 4    #The first is always size 2x1, the second will be randint(1,3) x randint(1,3)
        objects_cannot_be_occupied = 3 #The first is always size 2x1
        object_rate = 0.25

        n_rooms = random.randint(size - 3, size)
        rooms = fill_rooms(size, n_rooms) 
        person_positions = place_persons(size, rooms)

        while len(person_positions) == 0:
            n_rooms = random.randint(size - 3, size)
            rooms = fill_rooms(size, n_rooms) 
            person_positions = place_persons(size, rooms)

        person_genders = gender_persons(size)
        object_positions = place_objects(size, rooms, objects_can_be_occupied, objects_cannot_be_occupied, person_positions, object_rate)
        clues = generate_clues(size, rooms, person_positions, object_positions, person_genders)

        puzzle_text, clues_text = convert_to_clingo(size, rooms, person_genders, object_positions, clues)

        #print(puzzle_text)
        #print(clues_text)
        puzzle_file = os.path.join(output_dir+'/puzzles', f"puzzleA{size}-{seed}.lp")
        clues_file = os.path.join(output_dir+'/clues', f"cluesA{size}-{seed}.lp")

        os.makedirs(os.path.dirname(puzzle_file), exist_ok=True)
        os.makedirs(os.path.dirname(clues_file), exist_ok=True)
        
        with open(puzzle_file, "w") as f:
            f.write(puzzle_text)
        with open(clues_file, "w") as f:
            f.write(clues_text)

        end = time.time()
        time_lapsed = end - start
        print(f"Puzzle {seed} of size {size} created in {time_lapsed} seconds.")

            

Puzzle 0 of size 8 created in 9.450607061386108 seconds.
Puzzle 1 of size 8 created in 34.79714608192444 seconds.
Puzzle 2 of size 8 created in 87.98407697677612 seconds.
Puzzle 3 of size 8 created in 240.50743103027344 seconds.
Puzzle 4 of size 8 created in 2.112449884414673 seconds.
Puzzle 5 of size 8 created in 78.55894708633423 seconds.
Puzzle 6 of size 8 created in 42.07672905921936 seconds.
Puzzle 7 of size 8 created in 35.7421498298645 seconds.
